# EP241104a — HostPipeline quick-mode Aladin view

Run the cell below to recreate the interactive widget.

In [ ]:
import json
import astropy.units as u
from astropy.coordinates import Angle, SkyCoord
from astropy.table import Table
from ipyaladin import Aladin, EllipseError
from regions import CircleSkyRegion

config = json.loads('{"name": "EP241104a", "ra": 32.574, "dec": 31.555, "radius_arcsec": 190.0, "fov_deg": 0.025069444444444443, "survey": "CDS/P/PanSTARRS/DR1/color-z-zg-g", "candidates": [{"name": "2MASX J02101793+3131357", "ra": 32.574741, "dec": 31.526606, "r1": 18.4599991, "r2": 14.0296001, "pa": 80.0, "z": 0.0475756, "sep": 102.24368021936903}, {"name": "DESI J021008.79+313432.8", "ra": 32.536629, "dec": 31.575773, "r1": 9.905426, "r2": 4.6164794, "pa": 29.4647884, "z": 0.0459514, "sep": 136.86687119434492}, {"name": "DESI J021025.10+313047.4", "ra": 32.604599, "dec": 31.51318, "r1": 11.2991371, "r2": 1.6553459, "pa": 164.8301086, "z": 0.047559, "sep": 177.42926408677374}]}')
target = SkyCoord(config["ra"], config["dec"], unit="deg", frame="icrs")
aladin = Aladin(
    fov=config["fov_deg"],
    target=target,
    survey=config["survey"],
)

rows = config["candidates"]
if rows:
    cat_table = Table(
        rows=[
            (
                row["name"],
                row["ra"],
                row["dec"],
                row["z"],
                row["r1"],
                row["r2"],
                row["pa"],
                row["sep"],
            )
            for row in rows
        ],
        names=("Name", "RAJ2000", "DEJ2000", "z", "R1", "R2", "PA", "sep"),
    )
    cat_table["R1"].unit = u.arcsec
    cat_table["R2"].unit = u.arcsec
    cat_table["PA"].unit = u.deg
    cat_table["sep"].unit = u.arcsec
    aladin.add_table(
        cat_table,
        shape=EllipseError(
            maj_axis="R1",
            min_axis="R2",
            angle="PA",
            default_shape="cross",
        ),
        color="cyan",
    )
else:
    cat_table = Table(
        names=("Name", "RAJ2000", "DEJ2000", "z", "R1", "R2", "PA", "sep")
    )

search_circle = CircleSkyRegion(
    center=target,
    radius=Angle(config["radius_arcsec"], "arcsec"),
    visual={"edgecolor": "yellow", "linestyle": "dashed"},
)
aladin.add_graphic_overlay_from_region([search_circle])
aladin
